# Jupyter Model Factory

A broad Python ML/AI workbench covering classical ML, boosting, deep learning, embeddings, NLP, computer vision, reinforcement learning, model export, experiment tracking, FastAPI serving, vLLM inference, OpenAI-compatible clients, Anthropic, and LangGraph.

The pattern is consistent:

1. Use default data or bring your own JSON/CSV.
2. Train or run inference.
3. Evaluate/visualize.
4. Export an artifact when appropriate.
5. Register the artifact in a shared manifest.
6. Show how the model can be served or queried through an endpoint.

Heavy or cloud-hosted sections are optional.


## Optional installs

Install only what you need:

```bash
pip install numpy pandas matplotlib scikit-learn joblib xgboost
pip install lightgbm catboost tensorflow torch torchvision
pip install transformers sentence-transformers statsmodels opencv-python
pip install onnx onnxruntime skl2onnx mlflow
pip install fastapi uvicorn requests httpx
pip install openai anthropic langchain langgraph
pip install gymnasium stable-baselines3
```


In [ ]:
from pathlib import Path
import json
import os
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

MANIFEST_PATH = ARTIFACT_DIR / "model_manifest.json"

MODEL_MANIFEST = (
    json.loads(MANIFEST_PATH.read_text())
    if MANIFEST_PATH.exists()
    else {}
)

def register_artifact(name, framework, artifact, notes="", metrics=None):
    MODEL_MANIFEST[name] = {
        "framework": framework,
        "artifact": str(artifact),
        "notes": notes,
        "metrics": metrics or {},
    }
    MANIFEST_PATH.write_text(json.dumps(MODEL_MANIFEST, indent=2))
    return MODEL_MANIFEST[name]

print("Artifacts:", ARTIFACT_DIR.resolve())


## Data input

Choose:

- `default`
- `json`
- `csv`

Structured examples expect:

`age, income, transactions, credit_score, label`


In [ ]:
DATA_SOURCE = "default"

MY_JSON = '''
[
  {"age": 52, "income": 93000, "transactions": 31, "credit_score": 732, "label": 0},
  {"age": 34, "income": 48000, "transactions": 8, "credit_score": 581, "label": 1}
]
'''

CSV_PATH = "./data/my_data.csv"


In [ ]:
def load_structured_data():
    if DATA_SOURCE == "default":
        rng = np.random.default_rng(42)
        n = 1200

        df = pd.DataFrame({
            "age": rng.integers(18, 80, n),
            "income": rng.normal(80000, 25000, n).clip(15000),
            "transactions": rng.integers(0, 100, n),
            "credit_score": rng.integers(300, 850, n),
        })

        df["label"] = (
            (df["credit_score"] < 600)
            | (df["income"] < 45000)
            | ((df["transactions"] < 10) & (df["age"] < 30))
        ).astype(int)

        return df

    if DATA_SOURCE == "json":
        return pd.DataFrame(json.loads(MY_JSON))

    if DATA_SOURCE == "csv":
        return pd.read_csv(CSV_PATH)

    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

df = load_structured_data()
display(df.head())


## Matplotlib

**Best for:** exploratory plots, distributions, model comparisons, diagnostics.


In [ ]:
df["credit_score"].hist(bins=30)
plt.title("Credit Score Distribution")
plt.xlabel("credit_score")
plt.ylabel("count")
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ["age", "income", "transactions", "credit_score"]
TARGET = "label"

X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES],
    df[TARGET],
    test_size=0.2,
    random_state=42,
    stratify=df[TARGET],
)


# 1. scikit-learn — Logistic Regression

**Best for:** interpretable binary classification and strong linear baselines.

**Artifact:** `.joblib`


In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

logistic = LogisticRegression(max_iter=2000)
logistic.fit(X_train, y_train)
pred = logistic.predict(X_test)

metrics_logistic = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred, zero_division=0)),
    "recall": float(recall_score(y_test, pred, zero_division=0)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
}

path = ARTIFACT_DIR / "logistic_regression.joblib"
joblib.dump(logistic, path)
register_artifact("logistic_regression", "scikit-learn", path, metrics=metrics_logistic)

metrics_logistic


# 2. scikit-learn — Random Forest

**Best for:** nonlinear tabular classification/regression, robust baselines, feature importance.

**Artifact:** `.joblib`


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=10,
    random_state=42,
)

rf.fit(X_train, y_train)
pred = rf.predict(X_test)

metrics_rf = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
}

path = ARTIFACT_DIR / "random_forest.joblib"
joblib.dump(rf, path)
register_artifact("random_forest", "scikit-learn", path, metrics=metrics_rf)

metrics_rf


# 3. scikit-learn — Gradient Boosting

**Best for:** boosting baseline on structured data.

**Artifact:** `.joblib`


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
pred = gb.predict(X_test)

metrics_gb = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
}

path = ARTIFACT_DIR / "gradient_boosting.joblib"
joblib.dump(gb, path)
register_artifact("gradient_boosting", "scikit-learn", path, metrics=metrics_gb)

metrics_gb


# 4. scikit-learn — K-Means

**Best for:** unlabeled segmentation and exploratory clustering.

**Artifact:** `.joblib`


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES])

kmeans = KMeans(n_clusters=4, random_state=42, n_init="auto")
clusters = kmeans.fit_predict(X_scaled)

path = ARTIFACT_DIR / "kmeans.joblib"
joblib.dump({"scaler": scaler, "model": kmeans}, path)
register_artifact("kmeans", "scikit-learn", path)

pd.Series(clusters).value_counts().sort_index()


# 5. scikit-learn — PCA

**Best for:** dimensionality reduction, compression, visualization, preprocessing.

**Artifact:** `.joblib`


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

path = ARTIFACT_DIR / "pca.joblib"
joblib.dump({"scaler": scaler, "model": pca}, path)
register_artifact("pca", "scikit-learn", path)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df[TARGET], s=8)
plt.title("PCA Projection")
plt.show()


# 6. XGBoost

**Best for:** high-quality structured/tabular classification and regression.

**Artifact:** native `.json`


In [ ]:
try:
    from xgboost import XGBClassifier

    xgb = XGBClassifier(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=42,
    )

    xgb.fit(X_train, y_train)
    pred = xgb.predict(X_test)

    metrics_xgb = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred, zero_division=0)),
    }

    path = ARTIFACT_DIR / "xgboost_model.json"
    xgb.save_model(path)
    register_artifact("xgboost", "xgboost", path, metrics=metrics_xgb)

    print(metrics_xgb)

except ImportError:
    print("Install: pip install xgboost")


# 7. LightGBM

**Best for:** fast boosting on large structured datasets.

**Artifact:** model text file


In [ ]:
try:
    from lightgbm import LGBMClassifier

    lgbm = LGBMClassifier(
        n_estimators=250,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
    )

    lgbm.fit(X_train, y_train)
    pred = lgbm.predict(X_test)

    metrics_lgbm = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred, zero_division=0)),
    }

    path = ARTIFACT_DIR / "lightgbm_model.txt"
    lgbm.booster_.save_model(str(path))
    register_artifact("lightgbm", "lightgbm", path, metrics=metrics_lgbm)

    print(metrics_lgbm)

except ImportError:
    print("Install: pip install lightgbm")


# 8. CatBoost

**Best for:** structured data with categorical features and minimal preprocessing.

**Artifact:** `.cbm`


In [ ]:
try:
    from catboost import CatBoostClassifier

    cat = CatBoostClassifier(
        iterations=250,
        depth=6,
        learning_rate=0.05,
        verbose=False,
        random_seed=42,
    )

    cat.fit(X_train, y_train)
    pred = cat.predict(X_test).astype(int).ravel()

    metrics_cat = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred, zero_division=0)),
    }

    path = ARTIFACT_DIR / "catboost_model.cbm"
    cat.save_model(path)
    register_artifact("catboost", "catboost", path, metrics=metrics_cat)

    print(metrics_cat)

except ImportError:
    print("Install: pip install catboost")


# 9. TensorFlow / Keras

**Best for:** deep learning, production TensorFlow ecosystems, dense/CNN/RNN/transformer models.

**Artifact:** `.keras`


In [ ]:
try:
    import tensorflow as tf

    normalizer = tf.keras.layers.Normalization()
    normalizer.adapt(X_train.to_numpy(dtype="float32"))

    tf_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(len(FEATURES),)),
        normalizer,
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])

    tf_model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    tf_model.fit(
        X_train.to_numpy(dtype="float32"),
        y_train.to_numpy(dtype="float32"),
        epochs=8,
        batch_size=32,
        validation_split=0.2,
        verbose=0,
    )

    loss, accuracy = tf_model.evaluate(
        X_test.to_numpy(dtype="float32"),
        y_test.to_numpy(dtype="float32"),
        verbose=0,
    )

    path = ARTIFACT_DIR / "tensorflow_dense.keras"
    tf_model.save(path)

    register_artifact(
        "tensorflow_dense",
        "tensorflow/keras",
        path,
        metrics={"accuracy": float(accuracy)},
    )

    print({"accuracy": float(accuracy)})

except ImportError:
    print("Install: pip install tensorflow")


# 10. PyTorch MLP

**Best for:** custom deep learning, research, GPU training, transformers, flexible architectures.

**Artifact:** `.pt`


In [ ]:
try:
    import torch
    import torch.nn as nn

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    Xtr = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
    ytr = torch.tensor(y_train.to_numpy(), dtype=torch.long, device=device)
    Xte = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
    yte = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)

    mean = Xtr.mean(dim=0, keepdim=True)
    std = Xtr.std(dim=0, keepdim=True).clamp_min(1e-6)

    Xtrn = (Xtr - mean) / std
    Xten = (Xte - mean) / std

    torch_model = nn.Sequential(
        nn.Linear(len(FEATURES), 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 2),
    ).to(device)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(torch_model.parameters(), lr=0.001)

    for epoch in range(80):
        optimizer.zero_grad()
        logits = torch_model(Xtrn)
        loss = loss_fn(logits, ytr)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        pred = torch_model(Xten).argmax(dim=1)
        accuracy = (pred == yte).float().mean().item()

    path = ARTIFACT_DIR / "pytorch_mlp.pt"

    torch.save({
        "state_dict": torch_model.state_dict(),
        "mean": mean.cpu(),
        "std": std.cpu(),
        "features": FEATURES,
    }, path)

    register_artifact(
        "pytorch_mlp",
        "pytorch",
        path,
        metrics={"accuracy": float(accuracy)},
    )

    print({"accuracy": accuracy})

except ImportError:
    print("Install: pip install torch")


# 11. PyTorch Tiny CNN

**Best for:** images, handwriting, spatial features, computer vision.

This is a lightweight architecture/export example.

**Artifact:** `.pt`


In [ ]:
try:
    import torch
    import torch.nn as nn

    class TinyCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 8, 3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(8, 16, 3, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1)),
            )
            self.head = nn.Linear(16, 2)

        def forward(self, x):
            x = self.features(x)
            return self.head(x.flatten(1))

    cnn = TinyCNN()

    path = ARTIFACT_DIR / "pytorch_tiny_cnn.pt"
    torch.save(cnn.state_dict(), path)

    register_artifact(
        "pytorch_tiny_cnn",
        "pytorch",
        path,
        notes="Architecture/export example; not trained on a real image dataset.",
    )

    print(cnn)

except ImportError:
    print("Install: pip install torch")


# 12. Hugging Face Transformers

**Best for:** pretrained transformer inference/fine-tuning for classification, generation, summarization, QA, token classification.

This example uses a pretrained inference pipeline.


In [ ]:
try:
    from transformers import pipeline

    sentiment = pipeline("sentiment-analysis")
    result = sentiment("The model factory notebook is extremely useful.")
    print(result)

except Exception as exc:
    print("Optional section. Install: pip install transformers")
    print(exc)


# 13. sentence-transformers

**Best for:** embeddings, semantic search, RAG, clustering, similarity, recommendation.

Produces vectors rather than class labels.


In [ ]:
try:
    from sentence_transformers import SentenceTransformer

    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

    texts = [
        "Kubernetes model serving",
        "Machine learning pipelines",
        "Containerized inference APIs",
    ]

    vectors = embedding_model.encode(texts)

    print("shape:", vectors.shape)
    print(vectors[:2, :8])

except Exception as exc:
    print("Optional section. Install: pip install sentence-transformers")
    print(exc)


# 14. OpenCV

**Best for:** image preprocessing, OCR pipelines, traditional computer vision, cameras/video.

Not a training framework by itself; often sits before ML/DL models.


In [ ]:
try:
    import cv2

    image = np.zeros((128, 128, 3), dtype=np.uint8)
    image[32:96, 32:96] = 255

    blurred = cv2.GaussianBlur(image, (9, 9), 0)

    print("original:", image.shape)
    print("blurred:", blurred.shape)

except ImportError:
    print("Install: pip install opencv-python")


# 15. statsmodels

**Best for:** interpretable statistical models, econometrics, diagnostics, confidence intervals, hypothesis testing.

**Artifact:** pickle


In [ ]:
try:
    import statsmodels.api as sm

    X_sm = sm.add_constant(df[FEATURES])
    y_sm = df[TARGET]

    ols = sm.OLS(y_sm, X_sm).fit()

    path = ARTIFACT_DIR / "statsmodels_ols.pkl"
    ols.save(path)

    register_artifact(
        "statsmodels_ols",
        "statsmodels",
        path,
        notes="OLS demo on binary target for illustration.",
    )

    print(ols.params)

except ImportError:
    print("Install: pip install statsmodels")


# 16. Reinforcement Learning — Gymnasium + Stable-Baselines3

**Best for:** sequential decision-making, environments, control, robotics, policy optimization.

**Artifact:** Stable-Baselines3 model archive


In [ ]:
try:
    import gymnasium as gym
    from stable_baselines3 import PPO

    env = gym.make("CartPole-v1")

    rl_model = PPO("MlpPolicy", env, verbose=0)
    rl_model.learn(total_timesteps=2000)

    path = ARTIFACT_DIR / "ppo_cartpole"
    rl_model.save(str(path))

    register_artifact(
        "ppo_cartpole",
        "stable-baselines3/pytorch",
        path,
        notes="Small PPO example on CartPole.",
    )

    print("Saved:", path)

except ImportError:
    print("Install: pip install gymnasium stable-baselines3")


# 17. ONNX + ONNX Runtime

**Best for:** portable, cross-framework/cross-language inference and optimized serving.

This exports the Random Forest.

**Artifact:** `.onnx`


In [ ]:
try:
    from skl2onnx import to_onnx
    import onnxruntime as ort

    sample = X_train.iloc[:1].to_numpy(dtype=np.float32)

    onnx_model = to_onnx(
        rf,
        sample,
        target_opset=15,
    )

    path = ARTIFACT_DIR / "random_forest.onnx"
    path.write_bytes(onnx_model.SerializeToString())

    session = ort.InferenceSession(str(path))
    input_name = session.get_inputs()[0].name

    output = session.run(
        None,
        {input_name: sample},
    )

    register_artifact(
        "random_forest_onnx",
        "onnx",
        path,
        notes="Random Forest exported from scikit-learn.",
    )

    print([np.asarray(x).shape for x in output])

except ImportError:
    print("Install: pip install onnx onnxruntime skl2onnx")


# 18. MLflow

**Best for:** tracking parameters, metrics, artifacts, runs, and connecting experiments to model lifecycle workflows.


In [ ]:
try:
    import mlflow
    import mlflow.sklearn

    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("jupyter-model-factory")

    with mlflow.start_run(run_name="random-forest-demo"):
        mlflow.log_param("n_estimators", rf.n_estimators)
        mlflow.log_metric("accuracy", metrics_rf["accuracy"])
        mlflow.log_metric("f1", metrics_rf["f1"])
        mlflow.sklearn.log_model(rf, "model")

    print("Logged under ./mlruns")

except ImportError:
    print("Install: pip install mlflow")


# 19. FastAPI model serving

**Best for:** exposing trained Python models as HTTP endpoints.

This cell writes a complete FastAPI server file for the saved Random Forest.


In [ ]:
FASTAPI_SERVER = r"""
from pathlib import Path
import joblib
import pandas as pd

from fastapi import FastAPI
from pydantic import BaseModel, Field

MODEL_PATH = Path("artifacts/random_forest.joblib")
model = joblib.load(MODEL_PATH)

app = FastAPI(title="Model Factory API")

class PredictionRequest(BaseModel):
    age: float = Field(ge=0)
    income: float = Field(ge=0)
    transactions: float = Field(ge=0)
    credit_score: float = Field(ge=0)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/v1/predict")
def predict(request: PredictionRequest):
    row = pd.DataFrame([{
        "age": request.age,
        "income": request.income,
        "transactions": request.transactions,
        "credit_score": request.credit_score,
    }])

    prediction = int(model.predict(row)[0])
    probability = float(model.predict_proba(row)[0][1])

    return {
        "prediction": prediction,
        "label": "HIGH_RISK" if prediction == 1 else "LOW_RISK",
        "probability": probability,
        "probability_percent": round(probability * 100, 2),
    }
"""

server_path = ARTIFACT_DIR / "fastapi_model_server.py"
server_path.write_text(FASTAPI_SERVER)

print("Wrote:", server_path)
print("Run:")
print("uvicorn artifacts.fastapi_model_server:app --host 0.0.0.0 --port 8000")


In [ ]:
import requests

FASTAPI_URL = "http://127.0.0.1:8000/v1/predict"

payload = {
    "age": 52,
    "income": 93000,
    "transactions": 31,
    "credit_score": 732,
}

print("POST", FASTAPI_URL)
print(payload)

# Uncomment after server starts:
# response = requests.post(FASTAPI_URL, json=payload, timeout=10)
# print(response.json())


# 20. vLLM — LLM serving/inference

**What it does:** serves LLMs efficiently, usually on GPUs.

**Best for:** self-hosted LLM inference, batching, high throughput, OpenAI-compatible APIs.

vLLM is primarily an inference/serving engine, not a general model trainer.


In [ ]:
VLLM_BASE_URL = "http://127.0.0.1:8001/v1"
VLLM_MODEL = "YOUR_MODEL_NAME"

print(VLLM_BASE_URL, VLLM_MODEL)


In [ ]:
try:
    from openai import OpenAI

    vllm_client = OpenAI(
        base_url=VLLM_BASE_URL,
        api_key="not-needed-for-local-vllm",
    )

    print("Client configured.")

    # Uncomment when a vLLM server is running:
    #
    # response = vllm_client.chat.completions.create(
    #     model=VLLM_MODEL,
    #     messages=[
    #         {
    #             "role": "user",
    #             "content": "Explain Random Forest versus XGBoost."
    #         }
    #     ],
    # )
    #
    # print(response.choices[0].message.content)

except ImportError:
    print("Install: pip install openai")


# 21. OpenAI client

**Best for:** hosted LLM inference, structured outputs, text generation, embeddings, and reasoning.

Set `OPENAI_API_KEY` in the environment.

This is optional and does not run unless you uncomment the request.


In [ ]:
try:
    from openai import OpenAI

    if os.getenv("OPENAI_API_KEY"):
        openai_client = OpenAI()

        print("OPENAI_API_KEY detected. Client ready.")

        # Example:
        #
        # response = openai_client.responses.create(
        #     model="YOUR_OPENAI_MODEL",
        #     input="Explain the difference between XGBoost and PyTorch."
        # )
        # print(response.output_text)

    else:
        print("Set OPENAI_API_KEY to enable hosted OpenAI calls.")

except ImportError:
    print("Install: pip install openai")


# 22. Anthropic client

**Best for:** hosted Claude inference and tool-using/agentic workflows.

Set `ANTHROPIC_API_KEY` in the environment.

Optional section.


In [ ]:
try:
    import anthropic

    if os.getenv("ANTHROPIC_API_KEY"):
        anthropic_client = anthropic.Anthropic()

        print("ANTHROPIC_API_KEY detected. Client ready.")

        # Example:
        #
        # message = anthropic_client.messages.create(
        #     model="YOUR_CLAUDE_MODEL",
        #     max_tokens=300,
        #     messages=[
        #         {
        #             "role": "user",
        #             "content": "Explain this model result in plain English."
        #         }
        #     ],
        # )
        # print(message.content)

    else:
        print("Set ANTHROPIC_API_KEY to enable Anthropic calls.")

except ImportError:
    print("Install: pip install anthropic")


# 23. LangGraph — orchestration

**Best for:** coordinating tools, retrieval, ML endpoints, LLM calls, branching, loops, and stateful agent workflows.

This tiny example routes a request toward either an ML endpoint, retrieval, or an LLM.


In [ ]:
try:
    from typing import TypedDict
    from langgraph.graph import StateGraph, START, END

    class AgentState(TypedDict):
        query: str
        route: str

    def router(state: AgentState):
        q = state["query"].lower()

        if "risk" in q or "predict" in q:
            route = "ml"
        elif "document" in q or "policy" in q:
            route = "retrieval"
        else:
            route = "llm"

        return {
            "query": state["query"],
            "route": route,
        }

    graph = StateGraph(AgentState)
    graph.add_node("router", router)
    graph.add_edge(START, "router")
    graph.add_edge("router", END)

    app = graph.compile()

    print(app.invoke({
        "query": "Predict customer risk"
    }))

except ImportError:
    print("Install: pip install langgraph")


# 24. Compare exported/trained models


In [ ]:
comparison_rows = []

for name, info in MODEL_MANIFEST.items():
    row = {
        "name": name,
        "framework": info["framework"],
        "artifact": info["artifact"],
    }
    row.update(info.get("metrics", {}))
    comparison_rows.append(row)

manifest_df = pd.DataFrame(comparison_rows)
display(manifest_df)


# 25. Final artifact manifest

Every section that exports a model writes into:

```text
artifacts/model_manifest.json
```

This gives you a simple local model catalog that can later be replaced or synchronized with MLflow Model Registry, SageMaker Model Registry, Vertex AI Model Registry, Azure ML, or another catalog.


In [ ]:
print(MANIFEST_PATH.read_text())


# Suggested next experiments

Use the scratch area to expand this notebook with:

- SHAP explainability
- Optuna hyperparameter tuning
- Ray Tune
- Dask / Spark data preparation
- torchvision transfer learning
- Hugging Face fine-tuning
- LoRA/PEFT
- MLflow Model Registry
- model serving on FastAPI
- vLLM on Kubernetes
- LangGraph calling both ML and LLM endpoints
- pgvector / FAISS RAG
- TensorRT / TensorRT-LLM
- Triton Inference Server
